# 📓 Notebook 01 — YouTube Transcript Pipeline

**Project:** IncidentIQ — AI-powered Incident Intelligence for Firefighters  
**Goal of this notebook:** Build the data ingestion pipeline that transforms a YouTube video into a searchable knowledge base stored in ChromaDB.

## What this notebook covers
1. Install and import all required packages
2. Extract the transcript from a YouTube video using `youtube-transcript-api`
3. Split the transcript into chunks using LangChain's text splitter
4. Generate vector embeddings using OpenAI's embedding model
5. Store the embeddings in a local ChromaDB vector database
6. Run a first similarity search to verify everything works

## Why this matters
This pipeline is the **foundation of the entire RAG system**. Without a properly chunked and embedded transcript, the agent cannot answer questions accurately. Every other notebook depends on this one.

---

## 1. Install required packages
Install all dependencies needed for this notebook.  
Run this cell once — you can skip it on subsequent runs.

In [1]:
!pip install youtube-transcript-api langchain langchain-openai langchain-community chromadb python-dotenv -q


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


## 2. Import libraries and load environment variables
We use `python-dotenv` to load the OpenAI API key from a `.env` file.  
Never hardcode API keys directly in your code — always use environment variables.

In [2]:
import os
from dotenv import load_dotenv

# YouTube transcript extraction
from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound, TranscriptsDisabled

# LangChain components
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# ChromaDB
import chromadb

# Load environment variables from .env file
load_dotenv()

# Verify OpenAI API key is loaded
assert os.getenv("OPENAI_API_KEY"), "❌ OPENAI_API_KEY not found. Check your .env file."
print("✅ Environment variables loaded successfully.")

✅ Environment variables loaded successfully.


## 3. Define the YouTube URL
Set the YouTube URL you want to process.  
The pipeline extracts the video ID automatically from any standard YouTube URL format.  

**Test video:** Karel Lambert — *Murphy comes to Brussels* (High Rise Fire Case Study)  
This is a real firefighting incident presentation — perfect for our use case.

In [3]:
# ── Configuration ─────────────────────────────────────────────────────────────
YOUTUBE_URL = "https://www.youtube.com/watch?v=7OH5FEWWM_k"
CHROMA_PATH = "../chroma_db"          # Persistent storage path for ChromaDB
COLLECTION_NAME = "incidentiq"        # Name of the ChromaDB collection
CHUNK_SIZE = 500                       # Max characters per chunk
CHUNK_OVERLAP = 50                     # Overlap between chunks to preserve context
EMBEDDING_MODEL = "text-embedding-3-small"  # OpenAI embedding model — fast and cost-efficient


def extract_video_id(url: str) -> str:
    """
    Extract the YouTube video ID from a URL.
    Supports standard (watch?v=) and shortened (youtu.be/) formats.
    """
    if "v=" in url:
        return url.split("v=")[1].split("&")[0]
    elif "youtu.be/" in url:
        return url.split("youtu.be/")[1].split("?")[0]
    else:
        raise ValueError(f"Could not extract video ID from URL: {url}")


video_id = extract_video_id(YOUTUBE_URL)
print(f"✅ Video ID extracted: {video_id}")

✅ Video ID extracted: 7OH5FEWWM_k


## 4. Fetch the YouTube transcript
We use `youtube-transcript-api` to fetch the official subtitles from YouTube.  
This approach is **safe and fast** — it only downloads text, never audio or video.  

**Language priority:** English first, then Dutch, then French.  
If no transcript is available, the user receives a clear error message — no Whisper fallback needed for our use case.

In [14]:
import re

def clean_transcript(text: str) -> str:
    """
    Remove auto-generated noise tags and empty timestamps
    from YouTube transcripts before storing in ChromaDB.
    Empty timestamps like [00:03] [00:17] with no content
    between them add noise and reduce retrieval quality.
    """
    # Remove noise tags like [Music], [Applause] etc.
    text = re.sub(r'\[Music\]|\[Applause\]|\[Laughter\]|\[Cheering\]', '', text)
    # Remove empty timestamps — timestamp followed immediately by another timestamp
    text = re.sub(r'\[\d{2}:\d{2}\]\s*(?=\[\d{2}:\d{2}\])', '', text)
    # Fix multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def fetch_transcript(video_id: str, languages: list = ["en", "nl", "fr"]) -> tuple:
    """
    Fetch the transcript of a YouTube video using the new API (v0.7.0+).
    Returns (plain_text, timestamped_text).
    - plain_text: clean text for summaries and display
    - timestamped_text: text with [MM:SS] markers for accurate source citation in RAG
    Language priority: English → Dutch → French.
    """
    try:
        # New API since v0.7.0 — requires instantiation before calling fetch()
        api = YouTubeTranscriptApi()
        fetched = api.fetch(video_id, languages=languages)
        transcript_list = fetched.snippets
    except Exception as e:
        raise ValueError(
            f"Could not fetch transcript for video '{video_id}'.\n"
            "Check that the CC button is visible in YouTube.\n"
            f"Error: {str(e)}"
        )

    # Build plain text — used for display, summaries and LLM input
    plain_text = " ".join([t.text for t in transcript_list])

    # Build timestamped text — preserves [MM:SS] for accurate citations
    # Example: "[04:32] the command post must always be outside the building"
    timestamped_parts = []
    for t in transcript_list:
        mins = int(t.start // 60)
        secs = int(t.start % 60)
        timestamped_parts.append(f"[{mins:02d}:{secs:02d}] {t.text}")
    timestamped_text = " ".join(timestamped_parts)

    # Clean both versions — remove [Music], [Applause] etc.
    plain_text = clean_transcript(plain_text)
    timestamped_text = clean_transcript(timestamped_text)

    return plain_text, timestamped_text


plain_text, timestamped_text = fetch_transcript(video_id)

print(f"✅ Transcript fetched and cleaned successfully.")
print(f"   Total characters (plain):       {len(plain_text):,}")
print(f"   Total characters (timestamped): {len(timestamped_text):,}")
print(f"\n📄 First 500 characters preview:")
print("-" * 60)
print(plain_text[:500])

✅ Transcript fetched and cleaned successfully.
   Total characters (plain):       19,793
   Total characters (timestamped): 24,105

📄 First 500 characters preview:
------------------------------------------------------------
I got drawn in on sort of into the subject about what happens after the fire I just want to share some experience with you I learned so much by talking to other people that I meet and I make really longstanding connections pouring water onto an EV in thermal rway is like pouring water onto the roof of your house when your kitchen's on for you thank you gentlemen thank you for uh the invitation of being here I'm going to talk about a fiery HUD in a high-rise building in Brussels an old high-rise 


## 5. Split the transcript into chunks
Large texts cannot be stored as a single vector — we need to split them into smaller chunks.  

**Why chunking matters:**
- Embeddings work best on short, focused pieces of text
- Smaller chunks = more precise retrieval
- Overlap between chunks ensures context is not lost at boundaries

We use `RecursiveCharacterTextSplitter` which splits on natural boundaries (paragraphs → sentences → words) before splitting mid-word.

In [15]:
def split_transcript(text: str, video_id: str, source_url: str) -> list[Document]:
    """
    Split transcript text into overlapping chunks and wrap them as LangChain Documents.
    Each Document carries metadata (video_id, source URL) for later filtering and citation.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " "],  # Split on natural boundaries first
    )

    chunks = splitter.create_documents(
        texts=[text],
        metadatas=[{
            "video_id": video_id,
            "source": source_url,
            "chunk_size": CHUNK_SIZE,
        }]
    )
    return chunks


chunks = split_transcript(timestamped_text, video_id, YOUTUBE_URL)

print(f"✅ Transcript split into {len(chunks)} chunks.")
print(f"   Chunk size:    {CHUNK_SIZE} characters")
print(f"   Chunk overlap: {CHUNK_OVERLAP} characters")
print(f"\n📦 Example chunk (chunk 3):")
print("-" * 60)
print(chunks[2].page_content)
print(f"\n🏷️  Metadata: {chunks[2].metadata}")

✅ Transcript split into 54 chunks.
   Chunk size:    500 characters
   Chunk overlap: 50 characters

📦 Example chunk (chunk 3):
------------------------------------------------------------
event plenty of things went [01:27] the wrong way maybe quick quickly [01:29] introduced myself so I've been a [01:31] firefighter for 22 years both as a [01:34] career or as a volunteer firefighter so [01:36] I I work in um in the place where I live [01:39] as a volunteer Sergeant could be like [01:42] crew Commander here and I'm a major [01:44] which is the second highest rank in the [01:46] Belgian fire service uh in Brussels [01:49] where I'm chief of training and I'm uh [01:51] responsible

🏷️  Metadata: {'video_id': '7OH5FEWWM_k', 'source': 'https://www.youtube.com/watch?v=7OH5FEWWM_k', 'chunk_size': 500}


## 6. Initialize ChromaDB and the embedding model
ChromaDB is our local vector database — it stores text chunks as high-dimensional vectors.  

**Why ChromaDB?**
- Runs entirely locally — no external account or API needed
- Persists data between sessions via `PersistentClient`
- Free and open-source

**Why `text-embedding-3-small`?**
- OpenAI's most cost-efficient embedding model
- 1536 dimensions — high quality for semantic search
- Fast: embeds thousands of chunks in seconds

In [16]:
# Initialize the OpenAI embedding model
# This model converts text into numerical vectors that capture semantic meaning
embedding_model = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# Initialize ChromaDB with persistent storage
# PersistentClient saves the database to disk so data survives between sessions
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

print(f"✅ ChromaDB initialized at: {CHROMA_PATH}")
print(f"✅ Embedding model ready:   {EMBEDDING_MODEL}")

✅ ChromaDB initialized at: ../chroma_db
✅ Embedding model ready:   text-embedding-3-small


## 7. Store chunks in ChromaDB
We embed each chunk using the OpenAI model and store the resulting vectors in ChromaDB.  

**What happens under the hood:**
1. Each chunk of text is sent to the OpenAI embedding API
2. The API returns a vector of 1536 numbers representing the semantic meaning
3. ChromaDB stores both the vector and the original text for later retrieval

This process runs **once per video** — subsequent questions use the stored vectors, not the API.

In [17]:
def store_in_chromadb(
    chunks: list[Document],
    embedding_model: OpenAIEmbeddings,
    collection_name: str,
    chroma_path: str,
) -> Chroma:
    """
    Embed all chunks and store them in a ChromaDB collection.
    If the collection already contains this video, existing chunks are cleared first
    to avoid duplicate entries.
    Returns the Chroma vectorstore instance for immediate use.
    """
    vectorstore = Chroma(
        client=chromadb.PersistentClient(path=chroma_path),
        collection_name=collection_name,
        embedding_function=embedding_model,
    )

    # Check if this video is already stored — avoid duplicates
    existing = vectorstore.get(where={"video_id": chunks[0].metadata["video_id"]})
    if existing["ids"]:
        print(f"⚠️  Found {len(existing['ids'])} existing chunks for this video. Clearing...")
        vectorstore.delete(ids=existing["ids"])

    # Embed and store all chunks
    # LangChain handles batching automatically to avoid API rate limits
    vectorstore.add_documents(chunks)

    return vectorstore


print(f"⏳ Embedding {len(chunks)} chunks and storing in ChromaDB...")
print(f"   This may take 10-30 seconds depending on transcript length.")

vectorstore = store_in_chromadb(chunks, embedding_model, COLLECTION_NAME, CHROMA_PATH)

print(f"\n✅ All chunks stored successfully in ChromaDB.")
print(f"   Collection: '{COLLECTION_NAME}'")
print(f"   Total chunks stored: {len(chunks)}")

⏳ Embedding 54 chunks and storing in ChromaDB...
   This may take 10-30 seconds depending on transcript length.
⚠️  Found 54 existing chunks for this video. Clearing...

✅ All chunks stored successfully in ChromaDB.
   Collection: 'incidentiq'
   Total chunks stored: 54


## 8. Verify: Run a first similarity search
We test the pipeline by running a semantic search query against the stored vectors.  

**How similarity search works:**
1. The query is embedded using the same OpenAI model
2. ChromaDB computes the cosine similarity between the query vector and all stored vectors
3. The top-k most similar chunks are returned

This is the exact mechanism the RAG agent uses to answer questions.

In [18]:
# Test query — directly related to the Karel Lambert video content
TEST_QUERY = "What mistakes were made during the high rise fire incident in Brussels?"

# Retrieve top 3 most relevant chunks
results = vectorstore.similarity_search(TEST_QUERY, k=3)

print(f"🔍 Query: '{TEST_QUERY}'")
print(f"\n📋 Top {len(results)} most relevant chunks retrieved:")
print("=" * 60)

for i, doc in enumerate(results):
    print(f"\n[Chunk {i+1}]")
    print(f"Source: {doc.metadata.get('source', 'unknown')}")
    print(f"Content: {doc.page_content[:300]}...")
    print("-" * 40)

🔍 Query: 'What mistakes were made during the high rise fire incident in Brussels?'

📋 Top 3 most relevant chunks retrieved:

[Chunk 1]
Source: https://www.youtube.com/watch?v=7OH5FEWWM_k
Content: Belgium high-rise is [02:42] defined as over 25 M so uh below that we [02:48] typically have one staircase and above [02:50] we have two we have a water Riser which [02:53] is wet all the time we don't have dry [02:55] Rises anymore so we have a quite good [02:57] standard in Belgium when it comes t...
----------------------------------------

[Chunk 2]
Source: https://www.youtube.com/watch?v=7OH5FEWWM_k
Content: have plenty of [03:10] buildings who have been built before 72 [03:13] and this of course is such a case so to [03:17] go quickly through the presentation [03:19] we'll talk about where the fire happened [03:22] what tactics we intended to use and [03:25] which ones we end up using the problems [03:...
----------------------------------------

[Chunk 3]
Source: https://www.youtube.com

## 9. Pipeline summary and stats
Final overview of what was processed and stored.  
This cell gives a quick sanity check before moving to Notebook 02.

In [19]:
# Retrieve total count from ChromaDB collection
collection = chroma_client.get_collection(COLLECTION_NAME)
total_stored = collection.count()

print("=" * 60)
print("📊 PIPELINE SUMMARY")
print("=" * 60)
print(f"  Video URL:          {YOUTUBE_URL}")
print(f"  Video ID:           {video_id}")
print(f"  Transcript length:  {len(plain_text):,} characters")
print(f"  Chunks created:     {len(chunks)}")
print(f"  Chunks in ChromaDB: {total_stored}")
print(f"  Embedding model:    {EMBEDDING_MODEL}")
print(f"  ChromaDB path:      {CHROMA_PATH}")
print(f"  Collection name:    {COLLECTION_NAME}")
print("=" * 60)
print("✅ Pipeline complete — ready for Notebook 02: RAG Chain")

📊 PIPELINE SUMMARY
  Video URL:          https://www.youtube.com/watch?v=7OH5FEWWM_k
  Video ID:           7OH5FEWWM_k
  Transcript length:  19,793 characters
  Chunks created:     54
  Chunks in ChromaDB: 54
  Embedding model:    text-embedding-3-small
  ChromaDB path:      ../chroma_db
  Collection name:    incidentiq
✅ Pipeline complete — ready for Notebook 02: RAG Chain


## 10. Evaluating 

In [23]:
# ── Notebook 01 — Pipeline Quality Tests ──────────────────────────────────────
# This cell verifies that the transcript, chunks and embeddings are correct
# before moving to Notebook 02. Run all tests and check for any FAIL results.

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("=" * 60)
print("🧪 PIPELINE QUALITY TESTS")
print("=" * 60)

tests_passed = 0
tests_failed = 0

def check(name: str, condition: bool, detail: str = ""):
    global tests_passed, tests_failed
    if condition:
        tests_passed += 1
        print(f"  ✅ PASS — {name}")
    else:
        tests_failed += 1
        print(f"  ❌ FAIL — {name}")
    if detail:
        print(f"         {detail}")

# ── Test 1: Transcript length ─────────────────────────────────────────────────
# A meaningful video should have at least 2000 characters
check(
    "Transcript is long enough to contain useful content",
    len(plain_text) > 2000,
    f"Got {len(plain_text):,} characters — minimum expected: 2,000"
)

# ── Test 2: No noise tags remaining ───────────────────────────────────────────
# [Music] and similar tags should have been removed by clean_transcript()
check(
    "No [Music] or noise tags remaining in transcript",
    "[Music]" not in plain_text and "[Applause]" not in plain_text,
    "clean_transcript() should have removed all noise tags"
)

# ── Test 3: Timestamps present in timestamped text ────────────────────────────
# timestamped_text should contain [MM:SS] markers for source citation
import re
timestamp_count = len(re.findall(r'\[\d{2}:\d{2}\]', timestamped_text))
check(
    "Timestamps present in timestamped text",
    timestamp_count > 10,
    f"Found {timestamp_count} timestamps — minimum expected: 10"
)

# ── Test 4: Chunks created ────────────────────────────────────────────────────
# At least 10 chunks expected for a meaningful video
check(
    "Enough chunks created for good retrieval coverage",
    len(chunks) >= 10,
    f"Got {len(chunks)} chunks — minimum expected: 10"
)

# ── Test 5: Chunk size within bounds ──────────────────────────────────────────
# No chunk should exceed CHUNK_SIZE by more than 20%
oversized = [c for c in chunks if len(c.page_content) > CHUNK_SIZE * 1.2]
check(
    "No oversized chunks",
    len(oversized) == 0,
    f"{len(oversized)} chunks exceed {CHUNK_SIZE * 1.2:.0f} chars"
)

# ── Test 6: Metadata present on all chunks ────────────────────────────────────
# Every chunk needs video_id metadata for agent citation and filtering
missing_meta = [c for c in chunks if not c.metadata.get("video_id")]
check(
    "All chunks have metadata",
    len(missing_meta) == 0,
    f"{len(missing_meta)} chunks missing video_id metadata"
)

# ── Test 7: ChromaDB contains expected number of chunks ───────────────────────
stored_count = chroma_client.get_collection(COLLECTION_NAME).count()
check(
    "ChromaDB contains all chunks",
    stored_count >= len(chunks),
    f"ChromaDB: {stored_count} | Expected: {len(chunks)}"
)

# ── Test 8: Similarity search returns relevant results ────────────────────────
# Query with a topic clearly present in the Karel Lambert video
test_results = vectorstore.similarity_search(
    "high rise building fire command post", k=3
)
check(
    "Similarity search returns results",
    len(test_results) == 3,
    f"Got {len(test_results)} results — expected 3"
)

# ── Test 9: Retrieved chunks contain relevant keywords ────────────────────────
# At least 3 keywords from the expected topic should appear in retrieved chunks
combined = " ".join([r.page_content.lower() for r in test_results])
keywords = ["fire", "building", "floor", "high", "command", "smoke", "brussels"]
found_keywords = [k for k in keywords if k in combined]
check(
    "Retrieved chunks contain relevant content",
    len(found_keywords) >= 3,
    f"Keywords found: {found_keywords}"
)

# ── Test 10: LLM can answer a question from the retrieved context ──────────────
# Full end-to-end test — transcript → chunks → retrieval → LLM answer
print("\n⏳ Running end-to-end RAG test (LLM call)...")

test_results_llm = vectorstore.similarity_search(
    "Brussels fire incident lessons learned", k=5
)

context = "\n\n".join([r.page_content for r in test_results_llm])

question = "What lessons were learned from this fire incident?"
prompt = f"""You are a firefighting expert. Answer the question based on the context below.
Be concise but informative. Use the information present in the context.

Context:
{context}

Question: {question}
Answer:"""

response = llm.invoke(prompt)
answer = response.content.strip()

check(
    "LLM generates a meaningful answer from retrieved context",
    len(answer) > 50,
    f"Answer length: {len(answer)} characters"
)

print(f"\n  💬 LLM Answer:\n  {answer}")

# ── Final Results ──────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"📊 RESULTS: {tests_passed} passed | {tests_failed} failed")
if tests_failed == 0:
    print("✅ All tests passed — pipeline is ready for Notebook 02!")
else:
    print("⚠️  Fix the failing tests before moving to Notebook 02.")
print("=" * 60)

🧪 PIPELINE QUALITY TESTS
  ✅ PASS — Transcript is long enough to contain useful content
         Got 19,793 characters — minimum expected: 2,000
  ✅ PASS — No [Music] or noise tags remaining in transcript
         clean_transcript() should have removed all noise tags
  ✅ PASS — Timestamps present in timestamped text
         Found 539 timestamps — minimum expected: 10
  ✅ PASS — Enough chunks created for good retrieval coverage
         Got 54 chunks — minimum expected: 10
  ✅ PASS — No oversized chunks
         0 chunks exceed 600 chars
  ✅ PASS — All chunks have metadata
         0 chunks missing video_id metadata
  ✅ PASS — ChromaDB contains all chunks
         ChromaDB: 54 | Expected: 54
  ✅ PASS — Similarity search returns results
         Got 3 results — expected 3
  ✅ PASS — Retrieved chunks contain relevant content
         Keywords found: ['fire', 'building', 'floor', 'high', 'smoke']

⏳ Running end-to-end RAG test (LLM call)...
  ✅ PASS — LLM generates a meaningful answer fro

---

## ✅ What we built in this notebook

| Step | What | Why |
|------|------|-----|
| 1 | Extracted video ID from URL | Works with any YouTube URL format |
| 2 | Fetched transcript via `youtube-transcript-api` | Safe, fast — text only, no audio/video download |
| 3 | Added timestamps to each entry | Enables accurate source citation in answers |
| 4 | Split into 500-char chunks with 50-char overlap | Optimal chunk size for semantic search |
| 5 | Generated embeddings with `text-embedding-3-small` | Converts text to searchable vectors |
| 6 | Stored everything in ChromaDB | Persistent local vector database |
| 7 | Verified with a similarity search | Confirmed pipeline is working correctly |

